# Micro-Elise NN Video

Record a SolarSystemLander video with a static Micro-Elise network layout as a bottom overlay band.

In [ ]:
# cell: setup
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "dqn/src").exists() and (path / "hpo/src").exists() and (path / "nn_viz/src").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


def compute_live_node_scales(rollouts, percentile=95):
    percentile_label = f"p{percentile:g}"
    layers = [
        ("in", np.abs(rollouts.observations)),
        ("h1", rollouts.h1),
        ("h2", rollouts.h2),
    ]
    rows = []
    for layer, values in layers:
        scale = float(np.percentile(values, percentile))
        rows.append({"layer": layer, "max": float(np.max(values)), percentile_label: scale})
    return {row["layer"]: row[percentile_label] for row in rows}, pd.DataFrame(rows)


REPO_ROOT = repo_root()
for source_root in ["dqn/src", "hpo/src", "distillation/src", "nn_viz/src"]:
    sys.path.insert(0, str(REPO_ROOT / source_root))

from hpo.environments.solar_system_lander.env import DEFAULT_WORLD_MIX, EnvFactory
from hpo.evaluation.rendering.solar_system_lander import render_config
from nn_viz import (
    RolloutSpec,
    collect_activations,
    compute_activity_layout,
    load_student_network,
    record_network_overlay_video,
)

In [ ]:
# cell: load-micro-elise; requires: setup
CHECKPOINT_PATH = Path("G:/Meine Ablage/rl_lab/distillation/runs/solar_system_lander_10d_elise_stp_student_64x64_20260721T161456Z/student_checkpoint.pt")
RESULT_DIR = REPO_ROOT / "nn_viz" / "_local_runs" / "micro_elise_02"
VIDEO_DIR = RESULT_DIR / "videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

q_net = load_student_network(CHECKPOINT_PATH)
env_factory = EnvFactory("10d", world_mix=DEFAULT_WORLD_MIX)

In [ ]:
# cell: build-layout-quicktest; requires: load-micro-elise
LAYOUT_WORLDS = ["earth"]
LAYOUT_SEEDS = [0]

rollout_specs = [RolloutSpec(world, seed) for world in LAYOUT_WORLDS for seed in LAYOUT_SEEDS]
rollouts = collect_activations(q_net, env_factory, rollout_specs)
layout = compute_activity_layout(rollouts, q_net, top_edges_per_target=3, output_edges_per_target=10)

LIVE_NODE_SCALES, live_node_scale_summary = compute_live_node_scales(rollouts)
display(pd.DataFrame([
    {
        "worlds": ", ".join(LAYOUT_WORLDS),
        "seeds": ", ".join(str(seed) for seed in LAYOUT_SEEDS),
        "frames": rollouts.frame_count,
        "nodes": len(layout.nodes),
        "edges": len(layout.edges),
    }
]))
live_node_scale_summary

In [ ]:
# cell: build-layout; requires: load-micro-elise
rollout_specs = [RolloutSpec(world, seed) for world in DEFAULT_WORLD_MIX for seed in range(5)]
rollouts = collect_activations(q_net, env_factory, rollout_specs)
layout = compute_activity_layout(rollouts, q_net, top_edges_per_target=3, output_edges_per_target=10)

LIVE_NODE_SCALES, live_node_scale_summary = compute_live_node_scales(rollouts)
display(pd.DataFrame([
    {
        "frames": rollouts.frame_count,
        "nodes": len(layout.nodes),
        "edges": len(layout.edges),
    }
]))
live_node_scale_summary

In [ ]:
# cell: record-video; requires: build-layout-quicktest or build-layout
VIDEO_WORLD = "earth"
VIDEO_SEED = 0
LIVE_WINDOW_STEPS = 1
video_path = record_network_overlay_video(
    q_net,
    env_factory,
    layout,
    world=VIDEO_WORLD,
    seed=VIDEO_SEED,
    output_path=VIDEO_DIR / f"{VIDEO_WORLD}_seed_{VIDEO_SEED}_nn_overlay.mp4",
    render_cfg=render_config([VIDEO_WORLD], overlay=True, skin="photorealistic_eagle", scale=2),
    overlay_height_ratio=0.32,
    overlay_alpha=0.70,
    live_overlay=True,
    live_window_steps=LIVE_WINDOW_STEPS,
    live_node_scales=LIVE_NODE_SCALES,
)
video_path